# 01 - Baseline: logistic regression

**Kaggle score: 0.76** (submitted as `submission.csv`).

This is the first try. It turns each column into numbers and trains two standard models.
It scored *below* the simple "women live, men die" sample file (about 0.765).
The extra columns added noise more than signal. See `02_family_group_rule.ipynb` for the next approach.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Quick look at the data

In [2]:
print("Survival rate:", round(train.Survived.mean(), 3))
print(train.groupby("Sex").Survived.mean().round(2))
print(train.groupby("Pclass").Survived.mean().round(2))
train.isna().sum()

Survival rate: 0.384
Sex
female    0.74
male      0.19
Name: Survived, dtype: float64
Pclass
1    0.63
2    0.47
3    0.24
Name: Survived, dtype: float64


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Feature preparation
- Title from the name (Mr, Mrs, Miss, Master, Rare)
- Missing age filled with the typical age for that title
- Family size, travelling alone, has cabin, port
- Fill values come from **train only**, so test data does not leak in.

In [3]:
def get_title(names):
    t = names.str.extract(r",\s*([^\.]+)\.")[0]
    t = t.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return t.where(t.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

AGE_BY_TITLE = train.groupby(get_title(train.Name))["Age"].median()
FARE_MEDIAN = train.Fare.median()

def prepare(df):
    out = pd.DataFrame(index=df.index)
    out["Pclass"] = df.Pclass
    out["IsFemale"] = (df.Sex == "female").astype(int)
    title = get_title(df.Name)
    for t in ["Mr", "Mrs", "Miss", "Master", "Rare"]:
        out[f"Title_{t}"] = (title == t).astype(int)
    out["Age"] = df.Age.fillna(title.map(AGE_BY_TITLE))
    out["FamilySize"] = df.SibSp + df.Parch + 1
    out["IsAlone"] = (out.FamilySize == 1).astype(int)
    out["Fare"] = df.Fare.fillna(FARE_MEDIAN)
    out["HasCabin"] = df.Cabin.notna().astype(int)
    emb = df.Embarked.fillna("S")
    for p in ["C", "Q", "S"]:
        out[f"Embarked_{p}"] = (emb == p).astype(int)
    return out

X, y, X_test = prepare(train), train.Survived, prepare(test)
X.head()

,Pclass,IsFemale,Title_Mr,Title_Mrs,Title_Miss,Title_Master,Title_Rare,Age,FamilySize,IsAlone,Fare,HasCabin,Embarked_C,Embarked_Q,Embarked_S
0,3,0,1,0,0,0,0,22.0,2,0,7.2500,0,0,0,1
1,1,1,0,1,0,0,0,38.0,2,0,71.2833,1,1,0,0
2,3,1,0,0,1,0,0,26.0,1,1,7.9250,0,0,0,1
3,1,1,0,1,0,0,0,35.0,2,0,53.1000,1,0,0,1
4,3,0,1,0,0,0,0,35.0,1,1,8.0500,0,0,0,1


## Compare models (5-fold cross-validation)

In [4]:
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2, random_state=42),
}
print(f"Baseline (women live): {(X.IsFemale == y).mean():.3f}")
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=5, scoring="accuracy")
    print(f"{name:<20}: {s.mean():.3f} (+/- {s.std():.3f})")

Baseline (women live): 0.787
Logistic regression : 0.828 (+/- 0.027)


Random forest       : 0.826 (+/- 0.021)


## Train on all data and write the submission

In [5]:
model = models["Logistic regression"].fit(X, y)
sub = pd.DataFrame({"PassengerId": test.PassengerId, "Survived": model.predict(X_test)})
sub.to_csv("submission.csv", index=False)
print(len(sub), "rows,", f"{sub.Survived.mean():.0%} predicted survived")

418 rows, 40% predicted survived


## Result
- Cross-validation said **0.83**, but Kaggle gave **0.76**.
- So cross-validation on this small dataset is too hopeful. Don't trust it alone to pick a model.